<a href="https://colab.research.google.com/github/crignot/patent-law-predictive-dashboard/blob/main/notebooks/patent_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicting patent grants from the first office action

**Task.** On the day an application's *first* office action is mailed, estimate the probability that the application is *eventually granted* (1) rather than abandoned (0).

**Why this framing.** In the USPTO Office Action Research Dataset, `allowed_claims` means *this specific office action* indicates at least one allowable claim (USPTO variable tables, 2017 release). It is not the application's final outcome, and the dataset contains only rejection letters (non-final `CTNF` and final `CTFR`), not notices of allowance. So the final outcome is joined from the USPTO Patent Examination Research Dataset (PatEx, 2022 release).

## 1. Data

Download into `data_raw/` (not committed):
- `office_actions.csv` from the [Office Action Research Dataset](https://www.uspto.gov/ip-policy/economic-research/research-datasets/office-action-research-dataset-patents) (2017 release, `office_actions.csv.zip`)
- `application_data.csv` from [PatEx](https://www.uspto.gov/ip-policy/economic-research/research-datasets/patent-examination-research-dataset-public-pair) (2022 release, `application_data.csv.zip`)

The build script parses dates before sorting, keeps each application's earliest action, restricts to utility applications, and labels outcomes from PatEx status. Still-pending applications are kept out of training and evaluation.

In [ ]:
%cd ..
!python scripts/build_dataset.py

In [ ]:
import json, pandas as pd
print(json.dumps(json.load(open("reports/dataset_summary.json")), indent=2))

### Sanity check: `allowed_claims` vs. the final outcome

A first action with allowable claims is a strong signal, but most granted applications had *no* allowable claims in their first action, which is why it can't serve as the target.

In [ ]:
df = pd.read_parquet("data_raw/first_actions.parquet").dropna(subset=["granted"])
print(df["allowed_claims"].value_counts(dropna=False))
pd.crosstab(df["allowed_claims"], df["granted"], normalize="index").round(3)

## 2. Split, models, evaluation

- **Train** 2008–2013 · **Validation** 2014 · **Test** 2015–2017 (by first-action year; no shuffling).
- **No SMOTE.** Resampling before splitting leaks information and evaluates on an artificial class balance. Models are fit on the real balance so probabilities stay interpretable.
- Models: constant baseline (training grant rate), logistic regression, XGBoost (depth chosen on validation log loss, early stopping on validation).
- Ablations: XGBoost without citation features; without technology center.
- The decision threshold is chosen on validation (balanced accuracy) and applied unchanged to test.

In [ ]:
!python scripts/train_evaluate.py

In [ ]:
pd.read_csv("reports/model_comparison.csv").round(4)

In [ ]:
pd.read_csv("reports/slices_tech_center.csv").round(3)

In [ ]:
pd.read_csv("reports/error_examples.csv", dtype={"app_id": str})

## 3. What changed from the first version of this notebook

- The earlier version applied SMOTE **before** the train/test split and evaluated on the resampled data, so its metrics (including the previously reported AUC) were inflated and are not comparable.
- It used `allowed_claims` (a property of one office action) as if it were the final outcome, and kept the *last* action of a random row sample after sorting dates as strings.
- It read only the first 1,000,000 rows of the file, which is not a random sample.
- Features are now built by one shared function (`utils/features.py`) used by both training and the dashboard.